In [1]:
import pandas as pd

In [3]:
df = pd.read_excel('telematics_trips.xlsx')

In [4]:
df.head()

,trip_id,timestamp,lat,lon,speed_kmh,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,T003,13.05.2025 08:41:46,53.574561,9.994448,24.301420,-0.089931,-0.084742,9.829237,0.037493,-0.003997,-0.033762
1,T003,13.05.2025 08:32:43,53.546222,9.958473,50.886718,-0.421501,0.120669,9.783959,0.021199,-0.010631,-0.010813
2,T001,11.03.2025 08:19:07,53.521086,9.941093,14.247312,0.480223,0.015935,9.810027,0.014959,0.002670,-0.019313
3,T002,12.04.2025 08:29:16,53.560964,10.109916,9.474372,0.244356,0.062021,9.800611,-0.021922,0.002477,0.001997
4,T001,11.03.2025 08:13:51,53.534190,9.961272,46.591492,0.431632,-0.185665,9.719194,0.002099,0.025436,-0.020566


In [7]:
df.shape

(2095, 11)

In [6]:
df.dtypes

trip_id          str
timestamp        str
lat          float64
lon          float64
speed_kmh    float64
acc_x        float64
acc_y        float64
acc_z        float64
gyro_x       float64
gyro_y       float64
gyro_z       float64
dtype: object

In [8]:
df.isna().sum()

trip_id        0
timestamp      0
lat          189
lon          189
speed_kmh    189
acc_x         36
acc_y         36
acc_z         36
gyro_x         0
gyro_y         0
gyro_z         0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(25)

In [10]:
df.describe()

,lat,lon,speed_kmh,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
count,1906.000000,1906.000000,1906.000000,2059.000000,2059.000000,2059.000000,2095.000000,2095.000000,2095.000000
mean,53.552035,10.007386,33.262476,0.132577,0.005173,9.808818,-0.000085,0.001399,0.000749
std,0.016559,0.059673,31.242249,7.883446,0.195374,0.100283,0.030632,0.030633,0.030142
min,53.515590,9.937251,4.093620,-113.056686,-0.584199,9.498053,-0.111003,-0.131673,-0.101313
25%,53.538029,9.958338,16.050641,-0.284657,-0.125538,9.739607,-0.021153,-0.019161,-0.020053
50%,53.560042,9.985607,34.087537,0.005723,0.004800,9.807469,-0.000613,0.000668,0.001093
75%,53.564468,10.062836,45.076434,0.293239,0.135387,9.876287,0.020584,0.021812,0.020922
max,53.577496,10.121300,479.259290,116.902426,0.631551,10.155405,0.124537,0.097238,0.096245


In [12]:
df.value_counts(['trip_id'])

trip_id
T003       730
T002       730
T001       635
Name: count, dtype: int64

In [14]:
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d.%m.%Y %H:%M:%S')
df.dtypes

trip_id                 str
timestamp    datetime64[us]
lat                 float64
lon                 float64
speed_kmh           float64
acc_x               float64
acc_y               float64
acc_z               float64
gyro_x              float64
gyro_y              float64
gyro_z              float64
dtype: object

In [16]:
bef = len(df)
bef

2095

In [17]:
df = df.drop_duplicates()
aft = len(df)
aft

2070

In [18]:
print(bef-aft)

25


In [21]:
#check time order
df.groupby('trip_id')['timestamp'].apply(lambda s: s.is_monotonic_increasing)

trip_id
T001    False
T002    False
T003    False
Name: timestamp, dtype: bool

In [22]:
df = df.sort_values(['trip_id', 'timestamp']).reset_index(drop=True)
df.groupby('trip_id')['timestamp'].apply(lambda s: s.is_monotonic_increasing)

trip_id
T001    True
T002    True
T003    True
Name: timestamp, dtype: bool

In [23]:
df_cpy = df.copy()

In [26]:
df_cpy['dt'] = df_cpy.groupby('trip_id')['timestamp'].diff().dt.total_seconds()
df_cpy['dt'].value_counts(dropna=False)

dt
1.0    2067
NaN       3
Name: count, dtype: int64

In [27]:
df['gps_missing'] = df['lat'].isna()

In [29]:
df['run_id'] = (df["gps_missing"] != df.groupby("trip_id")["gps_missing"].shift()).cumsum()

In [30]:
gps_gap = (df[df["gps_missing"]].groupby(["trip_id", "run_id"]).size().rename("gap_seconds"))
gps_gap 

trip_id  run_id
T001     2         65
T002     5         65
T003     8         57
Name: gap_seconds, dtype: int64

In [32]:
#for acc
# Is accelerometer data missing in this row?
df["acc_missing"] = df["acc_x"].isna()

# Give each consecutive run of missing/not-missing a number
df["run_id"] = (df["acc_missing"] != df.groupby("trip_id")["acc_missing"].shift()).cumsum()

# Length of each missing run
acc_gaps = (df[df["acc_missing"]]
            .groupby(["trip_id", "run_id"])
            .size()
            .rename("gap_seconds"))
acc_gaps

trip_id  run_id
T001     2         3
         4         4
         6         4
         8         4
T002     11        2
         13        2
         15        4
         17        2
T003     20        2
         22        3
         24        2
         26        2
Name: gap_seconds, dtype: int64

In [33]:
s = pd.Series([5, 7, 7, 2])
print(s.shift())
print(s.diff())
print(s.cumsum())

0    NaN
1    5.0
2    7.0
3    7.0
dtype: float64
0    NaN
1    2.0
2    0.0
3   -5.0
dtype: float64
0     5
1    12
2    19
3    21
dtype: int64


In [34]:
MAX_FILL = 5
acc_cols = ['acc_x', 'acc_y', 'acc_z']

df['acc_missing'] = df['acc_x'].isna()
df['gps_missing'] = df['lat'].isna()

df['acc_run_id'] = (df['acc_missing'] != df.groupby('trip_id')['acc_missing'].shift()).cumsum()
df['gps_run_id'] = (df['gps_missing'] != df.groupby('trip_id')['gps_missing'].shift()).cumsum()

df['acc_gap_len'] = df.groupby('acc_run_id')['acc_missing'].transform('sum')
df['gps_gap_len'] = df.groupby('gps_run_id')['gps_missing'].transform('sum')

filled = df.groupby('trip_id')[acc_cols].transform(lambda s: s.interpolate())
short_gaps = df['acc_missing'] & (df['acc_gap_len'] <= MAX_FILL)
df.loc[short_gaps, acc_cols] = filled.loc[short_gaps, acc_cols]

df['gaps_gps_filled'] = df['gps_missing'] & (df['gps_gap_len'] > MAX_FILL)



In [35]:
print("Accelerometer values filled:", short_gaps.sum())
print("Accelerometer still missing:", df["acc_x"].isna().sum())
print("GPS rows flagged (long gaps):", df["gaps_gps_filled"].sum())

Accelerometer values filled: 34
Accelerometer still missing: 0
GPS rows flagged (long gaps): 187


In [37]:
#outliers
SPEED_MAX = 200
ACC_MAX = 20
speed_outliers = df['speed_kmh'] > SPEED_MAX
acc_outliers = (df['acc_x'].abs() > ACC_MAX) | (df['acc_y'].abs() > ACC_MAX)

print("Speed outliers:", speed_outliers.sum())
print("Accelerometer outliers:", acc_outliers.sum())

Speed outliers: 9
Accelerometer outliers: 17


**Note**
The steps in cleaning the data matters the most, now we found out the gaps and filled with values, interpolated and flagged and next we did the outliers. So the steps matters the most as it leads to wrong numbers to be filled without identifying the outliers before so find out the outliers and do the filling 